In [1]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

In [2]:
from qrc_bloqade.encodings import amp_encode, angle_encode
from qrc_bloqade.hamiltonian import rydberg 
from qrc_bloqade.readouts import ZReadout
from qrc_bloqade.solver import BloqadeSolver 
from qrc_bloqade.utils import(train_split, normalize, get_predictor, print_readout) 


In [26]:
def test_qrc_angle_encoding():
    n_sites = 4
    omega = 1.12
    V = 1.12
    time = 1.01

    # Create instances of the concrete classes
    hamiltonian = rydberg(n_sites, omega, V)
    encoder = angle_encode(n_sites)  # Use AngleEncoding
    readout = ZReadout(n_sites)
    predictor = get_predictor()
    #scaler = MinMaxScaler(feature_range=(-1, 1))
    #splitter = train_split

    # Generate synthetic data
    X, y = make_regression(n_samples=100, n_features=n_sites, noise=0.1, random_state=402)

    # Normalize data
    X_normalized, y_normalized, scaler_X, scaler_y = normalize(X, y)

    # Split data
    X_train, X_val, X_test, y_train, y_val, y_test = train_split(
        X_normalized, y_normalized, test_size=0.4, random_state=402
    )
    print(f"Training set size: {len(X_train)}")
    print(f"Test set size: {len(X_test)}")
    # --- QRC Workflow ---
    # Encode
    encoded_train = encoder.encode(X_train)
    encoded_test = encoder.encode(X_test)
    encoded_train = np.array(encoded_train)
    encoded_test = np.array(encoded_test)

    first_state = encoded_train[0]
    norm = np.sum(np.abs(first_state)**2)
    print(f"First state norm (should be close to 1.0): {norm}")
    print(f"n_sites (qubits): {n_sites}")
    print(f"Expected state size: {2**n_sites}")
    #print(f"Actual train state size: {len(encoded_train)}")
    #print(f"Actual test state size: {len(encoded_test)}")
    print(f"Encoded train shape: {encoded_train.shape}")
    print(f"Encoded test shape: {encoded_test.shape}")
    print(f"First encoded train sample shape: {encoded_train[0].shape}")
    print(f"First encoded test sample shape: {encoded_test[0].shape}")
 
    # Evolve (Dynamics)
    solver = BloqadeSolver()
    evolved_train = []
    evolved_test = []
    
    for sample in encoded_train:
        result = solver.simulate(
            sample, hamiltonian=hamiltonian.construct_hamiltonian(), 
            duration=time, steps=100, n_qubits=n_sites
        )
        evolved_train.append(result)
        
    for sample in encoded_test:
        result = solver.simulate(
            sample, hamiltonian=hamiltonian.construct_hamiltonian(), 
            duration=time, steps=100, n_qubits=n_sites
        )
        evolved_test.append(result)
    
    train_features = readout.measure(evolved_train, n_sites)
    test_features = readout.measure(evolved_test, n_sites)

    print(f"Train features shape: {train_features.shape}")
    print(f"Test features shape: {test_features.shape}")

    # Train
    predictor.fit(train_features, y_train)
    predictions = predictor.predict(test_features)

    # Inverse transform to get original scale
    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    mse = mean_squared_error(y_test, predictions)

    print(f"Test MSE (Angle Encoding): {mse}")
    assert mse >= 0  # Basic check

#

In [27]:
# Cell 3: Test function (Amplitude Encoding)
def test_qrc_amplitude_encoding():
    n_sites = 2  # Use only 2 qubits for easier amplitude encoding
    omega = 1.12
    V = 1.12
    time = 1.0

    # QRC components
    hamiltonian = rydberg(n_sites, omega, V)
    encoder = amp_encode(n_sites)  # Use AmplitudeEncoding
    readout = ZReadout(n_sites)
    predictor = get_predictor()
    
    # --- Data Preparation (for amplitude encoding) ---
    # Generate synthetic data with the correct number of features (2^n_sites)
    X, y = make_regression(n_samples=100, n_features=2**n_sites, noise=0.1, random_state=402)
    
    # For amplitude encoding, we need to normalize each sample to have L2 norm = 1
    X_normalized = []
    for sample in X:
        norm = np.linalg.norm(sample)
        X_normalized.append(sample / norm)  # Normalize each sample to unit length
    X_normalized = np.array(X_normalized)
    
    # Normalize y values using scikit-learn scaler
    scaler_y = MinMaxScaler(feature_range=(-1, 1))
    y_normalized = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
    
    # Split data
    X_train, X_val, X_test, y_train, y_val, y_test= train_split(
        X_normalized, y_normalized, test_size=0.4, random_state=402
    )
    
    # --- QRC Workflow ---
    # Encode
    encoded_train = encoder.encode(X_train)
    encoded_test = encoder.encode(X_test)
    print(f"n_sites (qubits): {n_sites}")
    print(f"Expected state size: {2**n_sites}")
    print(f"Actual train state size: {len(encoded_train)}")
    print(f"Actual test state size: {len(encoded_test)}")
    print(f"Encoded train size: {encoded_train.shape}")
    print(f"Encoded test size: {encoded_test.shape}")

    # Evolve
    solver = BloqadeSolver()
    evolved_train = []
    evolved_test = []
    
    for sample in encoded_train:
        result = solver.simulate(
            sample, hamiltonian=hamiltonian.construct_hamiltonian(), 
            duration=time, steps=100, n_qubits=n_sites
        )
        evolved_train.append(result)
        
    for sample in encoded_test:
        result = solver.simulate(
            sample, hamiltonian=hamiltonian.construct_hamiltonian(), 
            duration=time, steps=100, n_qubits=n_sites
        )
        evolved_test.append(result)

    # Measure
    train_features = readout.measure(evolved_train, n_sites)
    test_features = readout.measure(evolved_test, n_sites)
    
    # Predict
    predictor.fit(train_features, y_train)
    predictions = predictor.predict(test_features)

    # Inverse transform to get original scale
    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test_original = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    # Evaluate
    mse = mean_squared_error(y_test_original, predictions)
    print(f"Test MSE (Amplitude Encoding): {mse}")
    assert mse >= 0

In [28]:
if __name__ == "__main__":
    print("Running Angle Encoding Test...")
    test_qrc_angle_encoding()
    print("\nRunning Amplitude Encoding Test...")
    test_qrc_amplitude_encoding()

Running Angle Encoding Test...
Training set size: 60
Test set size: 20
First state norm (should be close to 1.0): 1.0000000000000002
n_sites (qubits): 4
Expected state size: 16
Encoded train shape: (60, 16)
Encoded test shape: (20, 16)
First encoded train sample shape: (16,)
First encoded test sample shape: (16,)
Train features shape: (60, 4)
Test features shape: (20, 4)
Test MSE (Angle Encoding): 5537.83738295669

Running Amplitude Encoding Test...
n_sites (qubits): 2
Expected state size: 4
Actual train state size: 60
Actual test state size: 20
Encoded train size: (60, 4)
Encoded test size: (20, 4)
Test MSE (Amplitude Encoding): 32356.142988814234
